# Task 10 — End-to-End MLOps Pipeline

**Objectif :** transformer le modèle de la Task 9 (maintenance prédictive) en un vrai service déployable (API + validation + containerisation), plutôt qu'un simple notebook.

**Modèle repris :** Random Forest de la Task 9, ré-entraîné à l'identique et exporté en `.pkl`.

## 1. Vue d'ensemble du pipeline

```
Task 9 (notebook)          Task 10 (service)
─────────────────          ──────────────────
  entraînement    ──export──▶  model.pkl
                              type_encoder.pkl
                                    │
                                    ▼
                          FastAPI (main.py)
                          + validation Pydantic
                          + frontend de démo
                                    │
                                    ▼
                          Dockerfile (containerisation)
                                    │
                                    ▼
                          GitHub Actions (CI : tests à chaque push)
```

## 2. Export du modèle

Le code complet est dans `train_and_export.py`

In [1]:
# on charge juste les artefacts déjà exportés pour vérifier qu'ils fonctionnent
import joblib

model = joblib.load("../app/model.pkl")
type_encoder = joblib.load("../app/type_encoder.pkl")

print(type(model).__name__, "chargé -", model.n_features_in_, "features attendues")
print("Classes Type :", dict(zip(type_encoder.classes_, type_encoder.transform(type_encoder.classes_))))

c:\Users\Administrateur\Desktop\ml-internship\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Administrateur\Desktop\ml-internship\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Administrateur\Desktop\ml-internship\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle est

RandomForestClassifier chargé - 6 features attendues
Classes Type : {'H': np.int64(0), 'L': np.int64(1), 'M': np.int64(2)}


## 3. L'API FastAPI (`app/main.py`)

Point clé demandé par l'énoncé : **validation stricte du schéma d'entrée avec Pydantic**. Un champ texte envoyé à la place d'un nombre, un type de machine inconnu, ou un champ manquant doivent être rejetés automatiquement — pas de plantage silencieux, pas de prédiction sur une donnée invalide.

In [2]:
# on teste l'API directement en mémoire (TestClient), sans avoir besoin de la lancer sur un port
import sys
sys.path.insert(0, "../app")
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

c:\Users\Administrateur\Desktop\ml-internship\venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
c:\Users\Administrateur\Desktop\ml-internship\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Administrateur\Desktop\ml-internship\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For mo

## 4. Test — cas valide

In [3]:
response = client.post("/predict", json={
    "type": "M",
    "air_temperature": 298.1,
    "process_temperature": 308.6,
    "rotational_speed": 1551,
    "torque": 42.8,
    "tool_wear": 0,
})

print(response.status_code)
response.json()

200


{'failure_predicted': False,
 'failure_probability': 0.0,
 'risk_level': 'faible'}

## 5. Test — cas à risque (valeurs extrêmes)

In [4]:
response_risk = client.post("/predict", json={
    "type": "L",
    "air_temperature": 303,
    "process_temperature": 312,
    "rotational_speed": 1400,
    "torque": 65,
    "tool_wear": 220,
})

print(response_risk.status_code)
response_risk.json()

200


{'failure_predicted': True, 'failure_probability': 0.69, 'risk_level': 'élevé'}

## 6. Test — la contrainte demandée : rejet des données invalides

In [5]:
cases = {
    "Texte dans un champ numérique": {
        "type": "M", "air_temperature": "chaud", "process_temperature": 308.6,
        "rotational_speed": 1551, "torque": 42.8, "tool_wear": 0,
    },
    "Type de machine inconnu": {
        "type": "X", "air_temperature": 298.1, "process_temperature": 308.6,
        "rotational_speed": 1551, "torque": 42.8, "tool_wear": 0,
    },
    "Champ manquant": {
        "type": "M", "air_temperature": 298.1,
    },
    "Couple négatif": {
        "type": "M", "air_temperature": 298.1, "process_temperature": 308.6,
        "rotational_speed": 1551, "torque": -10, "tool_wear": 0,
    },
}

for label, payload in cases.items():
    r = client.post("/predict", json=payload)
    status = "✅ rejeté correctement" if r.status_code == 422 else "❌ PROBLÈME"
    print(f"{label:35s} -> HTTP {r.status_code}  {status}")

Texte dans un champ numérique       -> HTTP 422  ✅ rejeté correctement
Type de machine inconnu             -> HTTP 422  ✅ rejeté correctement
Champ manquant                      -> HTTP 422  ✅ rejeté correctement
Couple négatif                      -> HTTP 422  ✅ rejeté correctement


→ Tous les cas invalides sont bien rejetés en 422 (l'équivalent FastAPI du 400 Bad Request demandé) — aucun n'atteint le modèle avec une donnée corrompue.

## 7. Suite de tests complète

8 tests automatisés dans `app/test_api.py`, ceux qu'on vient de dérouler manuellement plus quelques autres (healthcheck, frontend). Lancés automatiquement par GitHub Actions à chaque push (`.github/workflows/ci.yml`).

## 8. Containerisation (Docker)

⚠️ Docker n'était pas disponible dans l'environnement où ce notebook a été préparé — le `Dockerfile` suit la structure standard (image `python:3.11-slim`, copie des artefacts, `uvicorn` en CMD) mais **n'a pas pu être testé directement**. À vérifier en priorité chez toi :

```bash
cd app
docker build -t predictive-maintenance-api .
docker run -p 8000:8000 predictive-maintenance-api
```

## 9. Bonus — frontend de démo

Un formulaire HTML simple (`app/static/index.html`) servi directement par l'API sur `/` — permet de tester des valeurs de capteurs sans passer par `curl` ou Postman. Utilise `fetch()` vers `/predict` et affiche le niveau de risque avec un code couleur.

**Prochaines étapes possibles :**
- Tester réellement le Dockerfile une fois Docker disponible
- Ajouter un endpoint `/predict/batch` pour scorer plusieurs machines d'un coup (CSV entier)
- Faire pareil avec le modèle YOLO de la Task 8 une fois son mAP amélioré (upload d'image au lieu de JSON)